# 🚗 UK Traffic Accidents — Analysis & Predictive Model

**Author:** Abdel Rahman Khwaira  
**Dataset:** [UK Road Accidents 2022 — Kaggle](https://www.kaggle.com/datasets/abdulmannan/road-accidents-cav/data)  
**Tools:** Python · Pandas · NumPy · Scikit-learn · Seaborn · Matplotlib

## Objective
Analyze 144,000+ UK road accident records to identify patterns and build a classification model that predicts accident severity (Slight / Serious / Fatal).

## Structure
1. Import Libraries
2. Load & Explore Data
3. Data Cleaning
4. Feature Encoding
5. Classification Models (Decision Tree, KNN, Random Forest, AdaBoost)
6. Model Comparison & Results


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

## 2. Load & Explore Data

The dataset contains 144,000+ traffic accident records from the UK in 2022, with 21 features covering time, location, road conditions, weather, and accident severity.

In [ ]:
# Load dataset — update path as needed
df_accidents = pd.read_excel('accidents.xlsx')
df_accidents.head()

In [ ]:
df_accidents.info()

In [ ]:
df_accidents.describe()

In [ ]:
# Check missing values
df_accidents.isna().sum()

## 3. Data Cleaning

**Strategy:**
- `Carriageway_Hazards` had 95%+ null values → dropped entirely
- Remaining nulls in categorical columns → filled with the column's mode (most frequent value)


In [ ]:
# Drop column with 95%+ null values
del df_accidents['Carriageway_Hazards']

In [ ]:
# Fill remaining nulls with mode
df_accidents['Road_Surface_Conditions'] = df_accidents['Road_Surface_Conditions'].fillna('Dry')
df_accidents['Weather_Conditions'] = df_accidents['Weather_Conditions'].fillna('Fine no high winds')
df_accidents['Road_Type'] = df_accidents['Road_Type'].fillna('Single carriageway')

In [ ]:
# Confirm no nulls remain
df_accidents.isna().sum()

## 4. Feature Encoding

Machine learning models require numerical inputs. We use `LabelEncoder` to convert categorical columns into numeric values.

In [ ]:
label_encoder = LabelEncoder()
categorical_columns = [
    'Day_of_Week', 'Junction_Control', 'Junction_Detail', 'Accident_Severity',
    'Light_Conditions', 'Local_Authority_(District)', 'Police_Force',
    'Road_Surface_Conditions', 'Road_Type', 'Urban_or_Rural_Area',
    'Weather_Conditions', 'Vehicle_Type'
]

df_encoded = df_accidents.copy()
for col in categorical_columns:
    df_encoded[col] = label_encoder.fit_transform(df_encoded[col])

df_encoded.head()

In [ ]:
# Prepare features and target
X = df_encoded.drop(['Accident_Severity', 'Time', 'Accident_Index', 'Accident Date'], axis=1)
y = df_encoded['Accident_Severity']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}")

## 5. Classification Models

### 5.1 Decision Tree

A simple but powerful model for classification. We first train with default parameters, then tune using GridSearchCV.

In [ ]:
# Baseline Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred_dt):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Hyperparameter tuning with GridSearchCV
param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_leaf': [1, 2, 5, 10],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

grid_search = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5)
grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.2f}")

best_dt = grid_search.best_estimator_
y_pred_best_dt = best_dt.predict(X_test)
print(f"Tuned Accuracy: {accuracy_score(y_test, y_pred_best_dt):.2f}")

### 5.2 K-Nearest Neighbors (KNN)

KNN requires feature scaling since it relies on distances between points.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

print(f"KNN Accuracy: {accuracy_score(y_test, y_pred_knn):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))

ConfusionMatrixDisplay.from_estimator(knn, X_test_scaled, y_test)
plt.title("KNN Confusion Matrix")
plt.show()

### 5.3 Random Forest

An ensemble of decision trees. Generally more robust and accurate than a single tree.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

### 5.4 AdaBoost

A boosting algorithm that combines weak learners (shallow trees) sequentially to improve performance.

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50, random_state=42
)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)

print(f"AdaBoost Accuracy: {accuracy_score(y_test, y_pred_ada):.2f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_ada))

## 6. Model Comparison & Results

In [ ]:
results = {
    'Model': ['Decision Tree (baseline)', 'Decision Tree (tuned)', 'KNN', 'Random Forest', 'AdaBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_best_dt),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_ada)
    ]
}

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
print(results_df.to_string(index=False))

# Bar chart
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x='Model', y='Accuracy', palette='Blues_d')
plt.title('Model Accuracy Comparison')
plt.ylim(0.5, 1.0)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## Conclusions & Bias Notes

**Best Model:** Decision Tree (tuned) achieved **86% accuracy** in predicting accident severity across 3 classes (Slight, Serious, Fatal).

**Known Bias Issues:**
- The dataset is heavily imbalanced — the majority of accidents are "Slight", which inflates accuracy
- The model performs poorly on "Fatal" cases due to very few samples
- `LabelEncoder` on ordinal features like `Accident_Severity` may introduce unintended ordering

**Improvement Roadmap:**
- Apply SMOTE or class weighting to handle class imbalance
- Use `OrdinalEncoder` for ordinal features
- Try deep learning models for better handling of imbalanced classes
- Collect more recent data beyond 2022
